# Chapter 5, Notebook 2: The Bootstrap

Any number we compute from data, such as a model's accuracy on held-out data, would come out a little differently if we'd collected different training or held-out data. How much differently? This short notebook shows how to measure that with the **bootstrap**, a method that resamples the data we already have.

We'll use it to answer a question that comes up whenever you compare two models: if one model scores higher on held-out data, is it actually better, or did it just get lucky with which points were held out?

### By the end, you'll be able to

1. Explain why a number computed from limited data, including a model's accuracy on held-out data, is uncertain, and why that matters when comparing two models.
2. Describe a bootstrap sample: $n$ points drawn from the data at random, with replacement.
3. Describe the bootstrap procedure and compute a bootstrap standard error from the bootstrap values.
4. Read a bootstrap confidence interval for the difference between two models, and explain what it means when the interval includes zero.
5. Explain that we *assume* real data come from a fixed but unknown recipe, the **data-generating process**, and distinguish the **population** (everything the recipe could produce) from the **sample** (the data we actually have).
6. Explain that the methods in this course (the Bayes classifier's guarantee, cross-validation, and the bootstrap) are about **in-distribution** data, and say what **distribution shift** means.

## 1 · The problem: better, or just lucky? <a id="section-1"></a>

We reuse the simulated data from the last notebook: the same recipe, and the same 200 training points. Because we wrote the recipe, we can generate as much new data as we like. With real data we can't, do that.

We will:

- train two models on the same 200 training points, 
- measure each model's accuracy on 100 new points (50 per class): the **validation set**.

**Code worth reading.** Fit both models, then compute each one's accuracy on the 100 validation points. The cell also generates 50,000 fresh points from the recipe, but nothing about them is printed yet.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from ch_5_demos_helpers import simulate_data

sim = simulate_data()                # the same 200 training points as the last notebook
X, y = sim.X, sim.y

logistic = LogisticRegression(C=np.inf).fit(X, y)
knn = KNeighborsClassifier(n_neighbors=15).fit(X, y)

# The validation set: 100 new points (50 per class) from the same recipe.
val = simulate_data(n_train=50, n_test=0, seed=14)
X_val, y_val = val.X, val.y

# 50,000 fresh points from the recipe (25,000 per class). We hold these results back for now.
fresh = simulate_data(n_train=25000, n_test=0, seed=2026)
X_fresh, y_fresh = fresh.X, fresh.y

val_acc = {'Logistic regression': logistic.score(X_val, y_val),
           'KNN, K = 15': knn.score(X_val, y_val)}
fresh_acc = {'Logistic regression': logistic.score(X_fresh, y_fresh),
             'KNN, K = 15': knn.score(X_fresh, y_fresh)}

pd.DataFrame({'accuracy on 100 validation points': val_acc}).map('{:.0%}'.format)

We trained two models and measured their accuracy on 100 held-out points, the way you would in practice. Which one would you deploy?

**Ask:** Which model would you deploy, and how confident are you?

<details>
<summary>Answer</summary>

Logistic regression scored 2 percentage points higher on these 100 points, so it's the natural pick. How confident you should be depends on how much those scores would change if we had drawn a different 100 validation points, and a single set of 100 points can't show us that. The code below checks, by scoring both models on far more data.

</details>

### Demo A: Which model wins?

On simulated data, we can generate far more points than we'd ever have in practice. Let's see each model's accuracy on 50,000 fresh points from the same recipe. With that many points, the luck of the draw barely moves the scores, so these are close to each model's true accuracy on data from this recipe.

In [ ]:
pd.DataFrame({'accuracy on 100 validation points': val_acc,
              'accuracy on 50,000 fresh points': fresh_acc}).map('{:.0%}'.format)

Because we wrote the recipe, we can score both models on 50,000 fresh points from the recipe, and KNN comes out ahead. Choosing by the 100-point scores would have deployed the less accurate model.

Nobody made a mistake here. An accuracy measured on 100 points depends on which 100 points happened to be drawn: a different 100 would give somewhat different scores. Logistic regression's 2-percentage-point lead amounts to 2 points out of 100, and that can easily be luck in which points were drawn. Before trusting a number, or a comparison, we need to know how much it would change with a different sample.

With real data, we can't draw a fresh sample. That's the problem the bootstrap solves.

This is the same reason small gaps on a Kaggle leaderboard are often meaningless.

**A note on terms.** We used these 100 points to *choose* between models, so they're a **validation set**, the same idea as Monday's validation set approach. A **test set** is different: it's saved until the very end and used once, to score the final chosen model. You'll often see any held-out data loosely called a "test set," including in scikit-learn's `train_test_split`. Kaggle's public leaderboard is a test set that people end up using to choose models, which is exactly why small gaps on it can mislead.

# Bootstrapping Demo

Suppose we're testing an energy drink. Eight people try it, and each one rates how it changed their energy, from $-5$ (much worse) to $+5$ (much better). Those ratings are our data, shown on the top line below. The data has $n = 8$ points.

A **bootstrap sample** is a new data set of the same size, $n = 8$, made by drawing from our data **at random, with replacement**: pick one of the 8 people, copy their rating, put them back, and repeat until we've drawn 8. Because each person goes back after being drawn, some people show up more than once, and some not at all.

We then make $B$ bootstrap samples. Here $B = 20$. So $n$ is the size of *each* bootstrap sample, and $B$ is *how many* bootstrap samples we make.

- **Draw one** picks one person at random and copies their rating to the middle line.
- **Full sample** finishes the current bootstrap sample (or makes a whole new one), quickly.
- **All 20 samples** finishes every remaining bootstrap sample.
- When a bootstrap sample is complete, its mean is marked on the middle line, added to the table, and dropped onto the bottom line, which collects one mean per bootstrap sample.
- **Reset** starts over.

In [ ]:
from ch_5_demos_helpers import bootstrap_by_hand_demo

hand_demo = bootstrap_by_hand_demo(size_percent=75)

Each bootstrap sample gives a slightly different mean, because each one contains a different mix of people. The dots on the bottom line show how much the mean moves around from one bootstrap sample to the next.

**Ask:** After all $B = 20$ bootstrap samples, how could we measure how spread out the bootstrap means are, with a single number?

<details>
<summary>Answer</summary>

Take their **standard deviation**. That number is called the **bootstrap standard error** of the mean: an estimate of how much the mean would change if we could collect a new sample of 8 people. Section 2 writes this as a formula and has the computer make thousands of bootstrap samples.

</details>

## 1b · The recipe, the population, and the sample

In Demo A we could check the "truth" because we wrote the recipe that made the data. Statisticians call a recipe like this the **data-generating process**.

In the real world, we almost never know *the data-generating process*, or if one even exists. Instead, we *assume* our data came from one: some fixed process, unknown to us, that produced every row. (For regression, ISL writes this assumption as $Y = f(X) + \varepsilon$.) It's an assumption, not a fact about the world. Real processes drift and change over time, and future courses in machine learning will address these challenges.

Two words help here:

- The **population** is everything the data-generating process (aka "recipe") could produce.
- The **sample** is the data we actually have: when you actually collect data, consider that one set of draws from the recipe.

A number computed from the sample is called an **estimate**. Statisticians often write it as $\hat\theta$ ("theta-hat"), where $\theta$ is a placeholder for whatever number we care about: a mean, a median, or, here, a model's accuracy. Without the hat, $\theta$ means the same number for the whole population. (In other machine learning settings, you'll sometimes see $\theta$ used for a model's parameters instead; here it just means "the number we're estimating.")

- **Sampling variability:** A different sample would give a different $\hat\theta$. That variation is called **sampling variability**, and it's exactly what Demo A showed: a different set of 100 validation points could have picked a different winner.

**Ask:** In Demo A, what played the role of the population, and what played the role of the sample?

<details>
<summary>Answer</summary>

The recipe's 50,000 fresh points stood in for the population; the 100 validation points were one sample.

</details>

## 2 · The bootstrap: resample the data you have <a id="section-2"></a>

With real data, we can't draw new samples from the recipe, because we can't see it. So the bootstrap uses the next best thing: it treats **our sample as a stand-in recipe**, and draws new samples from that. Here is the procedure.

1. Draw a **bootstrap sample**: $n$ points from your data, at random, **with replacement**. Some points appear more than once, and some not at all.
2. Compute the statistic on it. We write $\hat\theta$ for the number computed from the data (a mean, a median, an accuracy, a difference), and $\hat\theta^{*b}$ for the same number computed on bootstrap sample $b$.
3. Repeat $B$ times.
4. The standard deviation of the $B$ values estimates the standard error.

In symbols, this is ISL's equation 5.8, written with $\theta$ in place of the book's $\alpha$:

$$
\mathrm{SE}_B(\hat\theta) = \sqrt{\frac{1}{B-1}\sum_{b=1}^{B}\Big(\hat\theta^{*b} - \bar\theta^{*}\Big)^2}
$$

<details>
<summary>Interpreting this equation</summary>

A star marks a number computed from a **bootstrap sample** instead of from the original data.

- $\hat\theta$ is the number computed from the original data.
- $\hat\theta^{*b}$ is the same number computed from bootstrap sample $b$. The $b$ tells you which of the $B$ bootstrap samples it came from.
- $\bar\theta^{*}$ is the average of the $B$ starred values, $\hat\theta^{*1}, \dots, \hat\theta^{*B}$.

So a star means "from a resampled dataset," and no star means "from the original data."

</details>

**What can we do with this Standard Error estimate?:**

An estimate, $\hat\theta$, on its own (such as $\hat\theta=$"87% accuracy")  doesn't say how much to trust it. The standard error does: it's the typical amount $\hat\theta$ would change from one sample to the next, *if we could keep collecting new samples* of the same size. That makes it useful in several ways.

- **Report how sure you are.** "87% accuracy, with a standard error of about 3 percentage points" tells the reader that a different set of 100 validation points could easily have given 84% or 90%. Without it, 87% looks more precise than it is.
- **Build a rough interval.** If the bootstrap values are roughly bell-shaped, about 95% of them fall within 2 standard errors of the center, so $\hat\theta \pm 2\,\mathrm{SE}_B(\hat\theta)$ is a rough 95% interval for $\theta$. Section 3 uses a different, more direct interval, but the idea is the same: a range of plausible values for $\theta$.
- **Judge whether a gap is real or noise.** When two models' scores differ by much less than about 2 standard errors of that difference, we can't tell from this data which one is better. That's the question Demo A raised, and Section 3 answers it.
- **Decide whether you need more data.** A larger sample gives a smaller standard error. 

### Demo B: The bootstrap for a mean and a median

The data here are 40 waiting times, in minutes. Waiting times are usually **right-skewed**: most waits are short, but a few are much longer. The median has no simple standard error formula, so this demo shows the bootstrap handling a case the usual formula can't.

In this demo, we:

- **Make the data.** Draw 40 waits at random from an **exponential distribution** with mean 10 minutes, a common model for waiting times. Short waits are the most likely, and longer ones get steadily rarer but never impossible.
- **Look at the skew.** Half of the 40 values are under 10 minutes, but the longest is 54. Those few long waits pull the mean (12.2 minutes) above the median (9.8 minutes). The mean isn't exactly 10, because these 40 values are just one sample.
- **Bootstrap two statistics.** Estimate the standard error of the mean, which has a formula, and of the median, which doesn't.
- **Check against the formula.** For the mean, compare the bootstrap's answer with the formula's.

**The plot below** shows our 40 waits as a histogram, with a tick for each one along the bottom. The black curve is the recipe itself, the exponential distribution the waits were drawn from. Dashed lines mark the sample's mean and median, just as the dashed line marked the data's mean in the energy drink demo; solid lines mark the recipe's true mean (10) and true median (about 6.9).

**New sample from the recipe** draws a fresh 40 waits from the same recipe. Click it a few times and watch the dashed lines move around the solid ones: that's sampling variability. Only the picture changes; the rest of the demo uses our original 40 waits.

In [ ]:
from ch_5_demos_helpers import waiting_times_demo

waits = np.random.default_rng(85).exponential(scale=10, size=40)   # 40 skewed measurements from a distribution whose mean is 10

waits_demo = waiting_times_demo(waits, scale=10, size_percent=75)

#### Bootstrap version

With real data, we'd have only our one sample and no recipe to draw from, so we couldn't click "New sample from the recipe." The bootstrap does the next best thing: it keeps our 40 waits fixed and draws new samples **from them**.

- **The top plot** shows our 40 waits in gray, with dashed lines at their mean and median. The current bootstrap sample is outlined in purple, with solid lines at *its* mean and median. There's no recipe curve: in the bootstrap, our sample plays the role of the recipe.
- **The strip below it** has a tick for each of our 40 waits, and a dot for each time the current bootstrap sample drew that wait. Some waits are drawn more than once, and some not at all.
- **The bottom plots** collect the mean and the median of every bootstrap sample so far. Their standard deviations, shown in the titles, are the bootstrap standard errors.

**One more** draws one bootstrap sample, **100 more** draws 100, and **All 10,000** finishes. Watch the standard errors settle down as the histograms fill in. This demo uses the same random draws as the code further down, so after all 10,000 samples its standard errors match that code's printout.

In [ ]:
from ch_5_demos_helpers import waiting_times_bootstrap_demo

boot_demo = waiting_times_bootstrap_demo(waits, B=10_000, seed=463, size_percent=75)

The code below does exactly what the demo above did, without the pictures.

**Code worth reading.** `rng.integers(0, n, size=n)` picks $n$ row numbers at random, with replacement. Indexing the data with them gives one bootstrap sample. We do this $B = 10{,}000$ times and record the mean and the median each time.

In [ ]:
rng = np.random.default_rng(463)
n, B = len(waits), 10_000

boot_means, boot_medians = [], []
for b in range(B):
    sample = waits[rng.integers(0, n, size=n)]       # n points, with replacement
    boot_means.append(sample.mean())
    boot_medians.append(np.median(sample))

# Below, ddof=1 specifies a "degrees of freedom" correction. The (B-1) in the SE formula above is this correction. You can ignore this, or look it up if your're curious.
se_formula = waits.std(ddof=1) / np.sqrt(n)           # the formula s / sqrt(n)
se_boot_mean = np.std(boot_means, ddof=1)             # standard deviation of the B bootstrap means
se_boot_median = np.std(boot_medians, ddof=1)         # standard deviation of the B bootstrap medians

print(f'Formula standard error of the mean:     {se_formula:.2f}')
print(f'Bootstrap standard error of the mean:   {se_boot_mean:.2f}')
print(f'Bootstrap standard error of the median: {se_boot_median:.2f}')

In [ ]:
from ch_5_demos_helpers import plot_bootstrap_stats

plot_bootstrap_stats(boot_means, boot_medians, se_boot_mean, se_boot_median);

For the mean, the bootstrap matches the formula from chapter 5 of ISL. For the median, there's no simple formula, but the bootstrap works exactly the same way. That's its power: it gives the standard error for almost any number you can compute, using a computer instead of algebra.

**Ask:** Why must the resampling be *with* replacement?

<details>
<summary>Answer</summary>

Without replacement, drawing $n$ points from $n$ points just gives back the original data, so every bootstrap sample would be identical.

</details>

**Key Takeaways:**

- **One sample can miss the truth.** Our sample's mean (12.2) and median (9.8) both landed above the recipe's true values (10 and 6.9). Each new sample from the recipe lands somewhere different: that's sampling variability.
- **The bootstrap measures that variability from one sample.** With no access to the recipe, resampling our 40 waits gave a standard error for the mean of 1.64, almost the same as the formula's 1.66.
- **It works where formulas don't.** The median has no simple standard error formula, but the bootstrap gave one (2.34) with exactly the same code.
- **A standard error tells you how much to trust an estimate, not how to fix it.** The bootstrap values center on *our* sample's mean and median, not the true ones. The standard error says how far off a sample of 40 typically is, but not which direction ours is off.
- **A standard error is in the same units as the estimate: here, minutes.** The mean of a sample of 40 waits typically lands about 1.6 minutes from the true mean, and the median about 2.3 minutes from the true median. Going out 2 standard errors from our estimates gives a rough range of plausible true values, worked out below.

**Turning the standard errors into ranges of minutes.**

Step 1: the numbers we need, all in minutes.

| | Our estimate, $\hat\theta$ | Bootstrap standard error, $\mathrm{SE}_B(\hat\theta)$ | True value (from the recipe) |
|---|---|---|---|
| Mean | 12.2 | 1.64 | 10 |
| Median | 9.8 | 2.34 | $10 \ln 2 \approx 6.9$ |

Step 2: the rough 95% range from Section 2 is our estimate plus or minus 2 standard errors:

$$
\hat\theta \pm 2\,\mathrm{SE}_B(\hat\theta)
$$

Step 3, the mean: our estimate is $\hat\theta = 12.2$ minutes, with $\mathrm{SE}_B(\hat\theta) = 1.64$ minutes.

$$
\begin{aligned}
2 \times 1.64 &= 3.28 \text{ minutes} \\
12.2 - 3.28 &= 8.92 \approx 8.9 \text{ minutes} \\
12.2 + 3.28 &= 15.48 \approx 15.5 \text{ minutes}
\end{aligned}
$$

So the true mean is plausibly between **8.9 and 15.5 minutes**. The true mean, 10 minutes, is inside this range.

Step 4, the median: our estimate is $\hat\theta = 9.8$ minutes, with $\mathrm{SE}_B(\hat\theta) = 2.34$ minutes.

$$
\begin{aligned}
2 \times 2.34 &= 4.68 \text{ minutes} \\
9.8 - 4.68 &= 5.12 \approx 5.1 \text{ minutes} \\
9.8 + 4.68 &= 14.48 \approx 14.5 \text{ minutes}
\end{aligned}
$$

So the true median is plausibly between **5.1 and 14.5 minutes**. The true median, about 6.9 minutes, is inside this range.

Both ranges contain the true values, but they're wide, about 6.6 minutes for the mean and 9.4 minutes for the median: 40 waits can't pin down the typical wait to better than a few minutes. The ranges are rough because $\pm 2$ standard errors assumes the bootstrap values are roughly bell-shaped, which fits the means better than the medians.

## 3 · Back to the problem: better, or just lucky? <a id="section-3"></a>

In Section 1, KNN scored 85% on the 100 validation points and logistic regression scored 87%. The *difference* between these metrics is:

$$
\hat\theta = \text{KNN's accuracy} - \text{logistic regression's accuracy} = 85\% - 87\% = -2 \text{ percentage points}.
$$

Notice that now, we're representing *the difference* (between the accuracy scores) itself as the estimated value, $\hat\theta$. If we had drawn a different 100 validation points, this gap would come out differently. With real data we couldn't just sample new points, so we use  **the bootstrap** to estimate how much this difference could change from sample to sample. It's the same procedure as Demo B, with the difference in place of the mean:

1. Draw a bootstrap sample: 100 points from the 100 validation points, with replacement.
2. Compute both models' *accuracies* on that bootstrap sample, and compute the difference, $\hat\theta^{*b}$.
3. Repeat $B = 2{,}000$ times.

### Demo C: How much could the gap change with different validation points?

**Code worth reading.** The models never change, so first we record, for each of the 100 validation points, whether each model predicted correctly. A model's accuracy on a bootstrap sample is then just the fraction of the resampled points it got right. Both models use the **same** resampled points in every round, so each difference compares them on the same data. At the end, `np.percentile` finds the middle 95% of the 2,000 difference estimates.

In [ ]:
correct_logistic = logistic.predict(X_val) == y_val      # True where logistic regression got the point right
correct_knn = knn.predict(X_val) == y_val                # True where KNN got the point right
n = len(y_val)

rng = np.random.default_rng(463)
B = 2_000
diffs = []
for b in range(B):
    idx = rng.integers(0, n, size=n)                         # same resampled points for both models, so the comparison is fair
    diffs.append(correct_knn[idx].mean() - correct_logistic[idx].mean())

observed = correct_knn.mean() - correct_logistic.mean()      # KNN minus logistic regression on the real 100 points
low, high = np.percentile(diffs, [2.5, 97.5])
print(f'95% bootstrap interval for KNN minus logistic regression: '
      f'from {100 * low:+.0f} to {100 * high:+.0f} percentage points')

In [ ]:
from ch_5_demos_helpers import plot_bootstrap_difference

plot_bootstrap_difference(diffs, observed, (low, high));

The interval includes zero, so with only these 100 validation points we can't tell which model is better. The 2-point gap that made logistic regression look better is well within the noise. This doesn't mean the models are equally good; it means 100 validation points aren't enough to say which is better.

We used the **percentile interval**: the middle 95% of the bootstrap values.

**Ask:** How could we get an interval narrow enough to tell which model is better?

<details>
<summary>Answer</summary>

Use a larger validation set. With more points, the accuracy changes less from one sample of validation points to the next, so the bootstrap differences spread out less and the interval gets narrower.

</details>

## 4 · Summary <a id="section-4"></a>

- Any number computed from data is an estimate; the bootstrap measures how much it would change with a different sample.
- It works by resampling the data with replacement, and it works for almost any statistic. 
- **Cross-validation helps pick good parameters (like picking k for knn) and get a sense of how model performance will generalize on unseen data. The bootstrap asks how sure we are about an estimated value/statistic.**
- Use a validation set (or cross-validation) to choose a model, and keep a test set for one final, honest score.

### What all of this assumes

Everything in this notebook, and in the recent knn and cross-validation notebook, assumes that any new data will come from the **same data-generating process** (aka "recipe") as the data we trained our model on. Data from that same process are said to be **in distribution**.

Data from a different or changed data-generating process is said to be **out of distribution** for our model. Some examples: a model trained on a military hospital's patients and used at a hospital mostly serving elderly patients. The general name for the change is **distribution shift**. This is an important topic in evaluating frontier model safety, since part of the training and safety evaluation distribution of many frontier LLM's includes an awareness of the test environment. By definition, the deployed setting will be **out of distribution.**

Every guarantee we've discussed only applies in-distribution. The Bayes classifier is the best possible on new data *from the same source*. Cross-validation estimates error on new data *from the same source*. A bootstrap interval measures sampling variability *within one recipe*. None of them tells you how a model will do when the recipe changes, and in real projects, that's often where models fail.

There's one more assumption, which both the bootstrap and cross-validation rely on: each row was drawn **independently** from the recipe. If rows are related, for example several rows from the same patient, or measurements taken one day after another, resampling individual rows breaks those connections, and both methods can look more confident than they should.